#### 📃 **Documentation**

##### 💡 **Basic Info**
- Script:     GetEnhancedMetadataPropertiesForSemanticModel.ipynb
- Purpose:    Script checks if Semantic Models were upgraded to Enhanced Metadata
- Author:     Paweł Wrona
- Version:    1.2
- Updated:    2026-02-28

##### 📋 **Requirements**
- List of Workspace IDs
- **Contributor+** access to Workspaces
- **Semantic Link Labs** package installed

In [ ]:
# -----------------------------------------------------------------------------
# Install Semantic Link Labs
# It is using Semantic Link under the hood, providing additional functionalities
# -----------------------------------------------------------------------------
%pip install semantic-link-labs

In [ ]:
# Import required libraries
from sempy.fabric import FabricRestClient
from sempy_labs import get_semantic_model_definition
import pandas as pd

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
workspace_ids = []

# -----------------------------------------------------------------------------
# Initialize client
# -----------------------------------------------------------------------------
client = FabricRestClient()

results = []

for ws_id in workspace_ids:
    # Get workspace name
    try:
        ws_info = client.get(f"v1.0/myorg/groups/{ws_id}")
        ws_info.raise_for_status()
        ws_name = ws_info.json().get("name")
    except Exception as ex:
        ws_name = None
        print(f"[WARN] Could not read workspace info for {ws_id}: {ex}")

    # List datasets in workspace
    try:
        ds_resp = client.get(f"v1.0/myorg/groups/{ws_id}/datasets")
        ds_resp.raise_for_status()
        datasets = ds_resp.json().get("value", [])
    except Exception as ex:
        print(f"[ERROR] Could not list datasets for workspace {ws_id}: {ex}")
        continue

    for ds in datasets:
        dataset_id = ds.get("id")
        dataset_name = ds.get("name", "Unnamed Dataset")

        if not dataset_id:
            print(f"[WARN] Skipping dataset without id in workspace {ws_id}")
            continue

        # Get semantic model definition (TMSL) and extract properties
        try:
            bim = get_semantic_model_definition(
                dataset_id,
                format="TMSL",
                workspace=ws_id,
                return_dataframe=False,
            )
            compat = bim.get("compatibilityLevel")
            ds_ver = bim.get("model", {}).get("defaultPowerBIDataSourceVersion")
        except Exception as ex:
            compat = None
            ds_ver = None
            print(f"[WARN] Could not read model definition for '{dataset_name}' ({dataset_id}): {ex}")

        results.append(
            {
                "workspace_id": ws_id,
                "workspace_name": ws_name,
                "semantic_model_id": dataset_id,
                "semantic_model_name": dataset_name,
                "compatibility_level": compat,
                "default_power_bi_data_source_version": ds_ver,
            }
        )

# Convert results to DataFrame with stable column order
columns = [
    "workspace_id",
    "workspace_name",
    "semantic_model_id",
    "semantic_model_name",
    "compatibility_level",
    "default_power_bi_data_source_version",
]
df = pd.DataFrame(results, columns=columns)

display(df)